# 7 — RAG Pipeline

This is where everything comes together.

**Retrieval-Augmented Generation** in one sentence: take a user question, find the relevant text, and give it to the LLM as context.

We'll compare:
- Asking the LLM **without** context (it has to guess)
- Asking the LLM **with** retrieved context (it can cite the source)

In [ ]:
from dotenv import load_dotenv
from mistralai import Mistral
import chromadb
import os

load_dotenv()

client = Mistral(api_key=os.getenv("mistral_api_key"))
model = os.getenv("mistral_llm_model")

## Build the vector store

Same as notebook 6 — chunk, embed, store. We need it in memory to query later.

In [ ]:
with open("sample_text.txt", "r") as f:
    text = f.read()

chunks = [p.strip() for p in text.split("\n\n") if p.strip()]

embeddings = client.embeddings.create(
    model="mistral-embed",
    inputs=chunks,
)
embeddings = [item.embedding for item in embeddings.data]

chroma_client = chromadb.PersistentClient(path="dbs")
try:
    chroma_client.delete_collection(name="chocolate_demo")
except:
    pass
collection = chroma_client.create_collection(name="chocolate_demo")
collection.add(
    documents=chunks,
    embeddings=embeddings,
    ids=[f"chunk_{i}" for i in range(len(chunks))],
)

print(f"Vector store ready ({collection.count()} chunks)")

## The question

In [ ]:
question = "What temperature should chocolate be stored at?"

## Without context — just the LLM

The model has to rely entirely on its training data. It might be right, it might hallucinate, it might be vague.

In [ ]:
response_no_context = client.chat.complete(
    model=model,
    messages=[{"role": "user", "content": question}],
)

print(response_no_context.choices[0].message.content)

## With context — RAG

Now let's do it properly:
1. Embed the question
2. Retrieve the most relevant chunks from ChromaDB
3. Inject them into the prompt
4. Let the LLM answer based on that context

In [ ]:
# step 1: embed the question
query_embedding = client.embeddings.create(
    model="mistral-embed",
    inputs=[question],
).data[0].embedding

# step 2: retrieve relevant chunks
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3,
)

retrieved_context = "\n\n".join(results["documents"][0])

print("Retrieved context:")
print("---")
print(retrieved_context)

In [ ]:
# step 3: build the prompt
prompt = f"""Based on the following context, answer the question.
If the context does not contain the answer, say so.

Context:
{retrieved_context}

Question: {question}"""

# step 4: send to the LLM
response_with_context = client.chat.complete(
    model=model,
    messages=[{"role": "user", "content": prompt}],
)

print(response_with_context.choices[0].message.content)

## The difference

- **Without context** — the LLM gives a generic answer from its training data. Might be right, might not. No way to verify.
- **With context** — the answer is grounded in the actual text. It can point to specifics ("15 to 18 degrees Celsius") because it's reading the source.

That's RAG. The LLM doesn't need to "know" everything — it just needs the right context at the right time.

## Try other questions

Change the question and re-run the cells above:

In [ ]:
question = "Who invented milk chocolate?"

# retrieve
query_embedding = client.embeddings.create(
    model="mistral-embed",
    inputs=[question],
).data[0].embedding

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=2,
)

retrieved_context = "\n\n".join(results["documents"][0])

# generate
prompt = f"""Based on the following context, answer the question.
If the context does not contain the answer, say so.

Context:
{retrieved_context}

Question: {question}"""

response = client.chat.complete(
    model=model,
    messages=[{"role": "user", "content": prompt}],
)

print(f"Q: {question}")
print(f"A: {response.choices[0].message.content}")